- import libraries: torch, torchvision
- load mnist dataset: transform it, convert it to batches
- set up nn model using nn.Module
- set up training loop and evaluation after every 1000 epoch

In [1]:
## Import necessary libraries
import torch
import torchvision

from torch import nn
from torch import optim
from torchvision.transforms import v2
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST

In [2]:
## make default device as cuda
device = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(device)
print(device)

cpu


In [3]:
## Download data and apply transform
transform = v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.0],std=[1.0]),
    ]
)

training_data = MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform,
)

testing_data = MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform,
)

100%|██████████| 9.91M/9.91M [00:00<00:00, 64.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.71MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 15.0MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.21MB/s]


In [4]:
print(training_data[0][0].shape) ## <<-- image
print(training_data[0][1]) ## <<-- label

torch.Size([1, 28, 28])
5


In [5]:
## Convert dataset into batches using dataloader
training_data_loader = DataLoader(
    dataset=training_data,
    batch_size=32,
    shuffle=True,
)

testing_data_loader = DataLoader(
    dataset=testing_data,
    batch_size=32,
    shuffle=False,
)

In [6]:
## Model
class MnistClassificationModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 500)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(500, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)

        return x


model = MnistClassificationModel().to(device)

In [7]:
## Loss Function and Optimizer
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    params=model.parameters(),
)

In [15]:
## Setup - training loop
for epoch in range(3):
    model.train()
    train_loss_sum = 0
    total_train_points = 0

    for train_batch_index, (train_input, train_target) in enumerate(training_data_loader):
        optimizer.zero_grad()

        train_input = train_input.to(device)
        train_target = train_target.to(device)

        train_output = model(train_input)
        train_loss = loss_function(train_output, train_target)

        batch_size = train_target.size(0)
        total_train_points += batch_size
        train_loss_sum += (train_loss.item() * batch_size)

        train_loss.backward()
        optimizer.step()

    if (epoch)%1 == 0:
        train_loss_avg = train_loss_sum/total_train_points
        print(f"epoch = {epoch} | train_loss_avg = {train_loss_avg}")

    if (epoch+1)%1 == 0:
        model.eval()
        test_loss_sum = 0
        total_test_points = 0

        with torch.no_grad():
            for test_batch_index,(test_input, test_target) in enumerate(testing_data_loader):

                test_input = test_input.to(device)
                test_target = test_target.to(device)

                test_output = model(test_input)
                test_loss = loss_function(test_output, test_target)

                batch_size = test_target.size(0)
                total_test_points += batch_size
                test_loss_sum += (test_loss.item() * batch_size)

            test_loss_avg = test_loss_sum/total_test_points
            print(f"<--->\nepoch = {epoch} | test_loss_avg = {test_loss_avg}\n<--->")


epoch = 0 | train_loss_avg = 0.05247700620504717
<--->
epoch = 0 | test_loss_avg = 0.07925098769026809
<--->
epoch = 1 | train_loss_avg = 0.03766681199236773
<--->
epoch = 1 | test_loss_avg = 0.06501860523417126
<--->
epoch = 2 | train_loss_avg = 0.027547457026713528
<--->
epoch = 2 | test_loss_avg = 0.08197093527988764
<--->
